# AAI-540 Group 2 — 30-Day Hospital Readmission Risk

**School:** University of San Diego, CA  
**Authors:** Jose Sandoval, Manikanta Katuri, Michael Domingo  
**Course:** AAI-540 — Machine Learning Operations

**Project summary:** Predict 30-day hospital readmission risk on CMS DE-SynPUF Medicare claims to help care teams target post-discharge interventions.  
The notebook trains a logistic-regression baseline and an XGBoost model with a patient-grouped 40 / 30 / 30 split, gated on AUC ≥ 0.75 before any deployment.  
End-to-end MLOps is delivered on Amazon SageMaker — Feature Store, Model Registry, real-time endpoint, Model Monitor drift checks, and CloudWatch alarms + dashboard — driven by GitHub Actions CI/CD.

What it covers, end to end:
1. Configuration
2. Data ingest (CMS DE-SynPUF from S3, or deterministic synthetic generator for development)
3. 30-day readmission labeling + prior-utilization features
4. **40 / 30 / 30** train / test / validation split (patient-grouped, time-aware) — per AAI-540 final-project requirement
5. EDA + class-imbalance check
   - 5b. **Feature engineering** — winsorise, bucket, encode (same code path as training)
   - 5c. **Feature Store** — register the curated features (offline + online stores)
   - 5d. **Offline-store via Athena** — training-set assembly + cohort SQL against the Glue-cataloged feature table
6. Baseline (logistic regression) + primary (XGBoost) training
7. Subgroup / fairness evaluation
8. Final test-set evaluation + AUC deploy gate
   - 8b. **Model Store** — package and register the trained model in the Model Registry
9. **SageMaker Pipelines** orchestration (Preprocess → Train → Evaluate → AUC gate → Register)
10. Real-time endpoint deployment
11. **Model Monitor** baselines + hourly schedule
12. **CloudWatch alarms + dashboard** (no email subscribers — alerts and metrics are reviewed in the AWS console)
13. Live inference test
14. Cleanup

All code under `src/readmit/` is the same code that CI/CD runs in production — this notebook just drives it.


## 0. Environment setup

Some SageMaker Studio kernels ship only `sagemaker-core` (the modular v3 SDK), which owns the `sagemaker` namespace but is missing `sagemaker.session`, `sagemaker.feature_store`, `sagemaker.get_execution_role`, etc. — everything this project uses.

The cell below checks for the classic v2 SDK and installs `requirements.txt` only if needed, so **kernel restart + run-all** is fully self-contained. It is idempotent: on a kernel that already has the right SDK it just prints the version and skips pip.

In [ ]:
import os, sys, subprocess

def _classic_sagemaker_ok() -> bool:
    """True only if the classic v2 SageMaker Python SDK is importable."""
    try:
        import sagemaker  # noqa: F401
        from sagemaker.session import Session  # noqa: F401
        return getattr(sagemaker, "__version__", "").startswith("2.")
    except Exception:
        return False


def _find_requirements_txt() -> str | None:
    start = globals().get('__vsc_ipynb_file__') or globals().get('__file__') or os.getcwd()
    d = os.path.abspath(os.path.dirname(start) if os.path.isfile(start) else start)
    for _ in range(6):
        cand = os.path.join(d, 'requirements.txt')
        if os.path.isfile(cand):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


if _classic_sagemaker_ok():
    import sagemaker
    print(f"sagemaker {sagemaker.__version__} already present — skipping install.")
else:
    print("Classic SageMaker SDK not found on this kernel — installing...")
    req = _find_requirements_txt()
    if req:
        print(f"  pip install -r {req}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req])
    else:
        print("  requirements.txt not found — installing minimum pins only")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "sagemaker>=2.220,<3.0", "s3fs>=2024.3.0",
        ])
    for _m in [k for k in list(sys.modules) if k == "sagemaker" or k.startswith("sagemaker.")]:
        del sys.modules[_m]
    import sagemaker
    print(f"Installed sagemaker {sagemaker.__version__}")

from sagemaker.session import Session as _Session  # noqa: F401
from sagemaker.feature_store.feature_group import FeatureGroup as _FG  # noqa: F401
print("Classic SageMaker SDK surface OK (session + feature_store importable).")


## 1. Configuration

In [ ]:
import os, sys

def _bootstrap_readmit_path():
    """Find `src/readmit` by walking up from cwd and the notebook dir."""
    candidates = [os.getcwd()]
    nb_dir = globals().get('__vsc_ipynb_file__') or globals().get('__file__')
    if nb_dir:
        candidates.append(os.path.dirname(os.path.abspath(nb_dir)))
    seen = set()
    for start in candidates:
        d = os.path.abspath(start)
        for _ in range(6):
            if d in seen:
                break
            seen.add(d)
            src = os.path.join(d, 'src')
            if os.path.isdir(os.path.join(src, 'readmit')):
                if src not in sys.path:
                    sys.path.insert(0, src)
                return src
            parent = os.path.dirname(d)
            if parent == d:
                break
            d = parent
    return None


_resolved = _bootstrap_readmit_path()
if _resolved is None:
    raise RuntimeError(
        "Could not locate src/readmit on disk. Run `pip install -e .` from the "
        "project root, or set PROJECT_ROOT explicitly and re-run this cell."
    )
print(f'readmit package path: {_resolved}')

# AWS settings — resolved from the live SageMaker / STS session in the next cell.
# Override here only for cross-account runs.
BUCKET                  = None
SAGEMAKER_ROLE_ARN      = None
PROJECT_PREFIX          = 'readmit'

# Data source:
#   'cms-open'  -> curate public CMS DE-SynPUF (OMOP) into S3 (default)
#   'synthetic' -> in-memory generator (used by CI)
#   's3'        -> read pre-curated encounters.parquet from S3_CURATED_URI
DATA_SOURCE             = 'cms-open'
CMS_TIER                = '100k'         # '1k' (smoke) or '100k' (full demo)
S3_CURATED_URI          = None           # auto-derived from BUCKET in next cell
N_PATIENTS              = 50_000         # cap on patients (None = all)
FORCE_RECURATE          = False          # True ignores the cached parquet

ENDPOINT_NAME           = 'readmit-risk-dev'

FEATURE_GROUP_NAME      = 'readmit-encounter-features'
MODEL_PACKAGE_GROUP     = 'ReadmitRiskModels'

from readmit.config import TRAIN_FRAC, TEST_FRAC, VAL_FRAC
print(f'Split locked at train={TRAIN_FRAC} test={TEST_FRAC} val={VAL_FRAC}')


In [ ]:
import json, logging, os, sys
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
# Silence chatty third-party INFO logs (matplotlib emits 'Using categorical units...'
# on every string-x-axis plot; boto/sagemaker emit per-API-call lines).
for _noisy in ('matplotlib', 'boto3', 'botocore', 'urllib3', 's3transfer', 'sagemaker'):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

# First-run install (uncomment, run once, re-comment):
# PROJECT_ROOT = os.path.abspath(os.path.join(_resolved, os.pardir))
# !pip install -q -r {PROJECT_ROOT}/requirements.txt && pip install -q -e {PROJECT_ROOT}

if '_resolved' in globals() and _resolved and _resolved not in sys.path:
    sys.path.insert(0, _resolved)

from readmit.config import FEATURES, LABEL_COL, AUC_THRESHOLD_DEPLOY
from readmit.data.ingest   import load_encounters
from readmit.data.labeling import attach_labels_and_priors
from readmit.data.splits   import split_train_test_val
from readmit.features.engineering import add_derived_columns, winsorize_utilization
from readmit.models.evaluate import compute_metrics, recall_at_k, subgroup_metrics
from readmit.models.train    import build_xgb_pipeline, build_logreg_pipeline

import boto3
import sagemaker

# Session + region
try:
    sess = sagemaker.Session()
    AWS_REGION = sess.boto_region_name
except Exception as exc:
    logging.warning('sagemaker.Session() failed (%s); falling back to boto3.', exc)
    sess       = None
    AWS_REGION = boto3.session.Session().region_name or 'us-east-1'

# Account id via STS
_sts = boto3.client('sts', region_name=AWS_REGION)
_caller = _sts.get_caller_identity()
ACCOUNT_ID = _caller['Account']
_caller_arn = _caller['Arn']
logging.info('STS caller: %s', _caller_arn)

# Execution role — try sagemaker.get_execution_role(), then fall back to the
# IAM role behind the assumed-role STS principal.
if SAGEMAKER_ROLE_ARN is None:
    try:
        SAGEMAKER_ROLE_ARN = sagemaker.get_execution_role()
    except Exception as exc:
        logging.info(
            'sagemaker.get_execution_role() unavailable (%s); deriving from STS.', exc,
        )
        if ':assumed-role/' in _caller_arn:
            role_name = _caller_arn.split(':assumed-role/', 1)[1].split('/', 1)[0]
            SAGEMAKER_ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{role_name}'
        elif ':role/' in _caller_arn:
            SAGEMAKER_ROLE_ARN = _caller_arn
        else:
            SAGEMAKER_ROLE_ARN = os.environ.get('SAGEMAKER_ROLE_ARN')

# Default S3 bucket
if BUCKET is None:
    if sess is not None:
        try:
            BUCKET = sess.default_bucket()
        except Exception as exc:
            logging.warning('sess.default_bucket() failed (%s); deriving from STS.', exc)
    if BUCKET is None:
        BUCKET = f'sagemaker-{AWS_REGION}-{ACCOUNT_ID}'

# Fail-fast verification
_s3 = boto3.client('s3', region_name=AWS_REGION)
try:
    _s3.head_bucket(Bucket=BUCKET)
except Exception as exc:
    raise RuntimeError(
        f"BUCKET={BUCKET!r} is not reachable from this account/region ({exc}). "
        "Check that the SageMaker default bucket exists, or override BUCKET in "
        "the Configuration cell."
    )
if SAGEMAKER_ROLE_ARN is None:
    logging.warning(
        'SAGEMAKER_ROLE_ARN could not be resolved — Feature Store, Model '
        'Registry, Pipelines, Endpoint, Monitor and Alerts cells will skip.'
    )

if S3_CURATED_URI is None and DATA_SOURCE in ('cms-open', 's3'):
    S3_CURATED_URI = f's3://{BUCKET}/{PROJECT_PREFIX}/curated/'

print(f'AWS account       : {ACCOUNT_ID}')
print(f'AWS region        : {AWS_REGION}')
print(f'Execution role    : {SAGEMAKER_ROLE_ARN or "(not resolved — AWS cells will skip)"}')
print(f'Default S3 bucket : {BUCKET}')
print(f'Curated data URI  : {S3_CURATED_URI}')


## 2. Ingest encounters

In [ ]:
# For source='cms-open': reads the public OMOP tables from s3://synpuf-omop/,
# joins + derives the 14 columns the pipeline uses, writes the curated frame
# to S3_CURATED_URI + 'encounters.parquet', and caches it for subsequent runs.

encounters = load_encounters(
    source=DATA_SOURCE,
    s3_uri=S3_CURATED_URI,
    n_patients=N_PATIENTS,
    seed=42,
    cms_tier=CMS_TIER,
    force_recurate=FORCE_RECURATE, # True for first run, False for subsequent runs
)

print(f'Loaded {len(encounters):,} encounters across {encounters.beneficiary_id.nunique():,} patients')
print(f'Source           : {DATA_SOURCE}'
      + (f'  |  tier: {CMS_TIER}' if DATA_SOURCE == "cms-open" else '')
      + (f'  |  cached at: {S3_CURATED_URI}encounters.parquet' if S3_CURATED_URI else ''))
print(f'Admission window : {encounters.admission_date.min().date()}  →  {encounters.admission_date.max().date()}')
print('\nFirst 5 rows:')
encounters.head(5)


## 3. Attach 30-day readmission labels + recompute prior-utilization features

In [ ]:
labeled = attach_labels_and_priors(encounters)
rate = labeled[LABEL_COL].mean()
print(f'30-day readmission rate: {rate:.4f}  ({labeled[LABEL_COL].sum():,} positives / {len(labeled):,} rows)')
labeled[['beneficiary_id','admission_date','discharge_date','length_of_stay',
         'primary_diagnosis','prior_inpatient_90d','readmitted_30d']].head()

## 4. Patient-grouped 40 / 30 / 30 split

In [ ]:
train_df, test_df, val_df = split_train_test_val(labeled, seed=42)

summary = pd.DataFrame({
    'fold':       ['train','test','val'],
    'rows':       [len(train_df), len(test_df), len(val_df)],
    'patients':   [train_df.beneficiary_id.nunique(), test_df.beneficiary_id.nunique(),
                   val_df.beneficiary_id.nunique()],
    'pos_rate':   [train_df[LABEL_COL].mean(), test_df[LABEL_COL].mean(), val_df[LABEL_COL].mean()],
})
summary['row_pct']     = summary['rows']     / summary['rows'].sum()
summary['patient_pct'] = summary['patients'] / summary['patients'].sum()
summary

## 5. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labeled.groupby('primary_diagnosis')[LABEL_COL].mean().sort_values().plot(
    kind='barh', ax=axes[0], title='Readmission rate by diagnosis chapter')
labeled['length_of_stay'].hist(bins=30, ax=axes[1])
axes[1].set_title('Length of stay distribution'); axes[1].set_xlabel('days')
plt.tight_layout(); plt.show()

## 5b. Feature engineering

The same `add_derived_columns` + `winsorize_utilization` + `build_preprocessor` pipeline that `src/readmit/models/train.py` invokes during SageMaker training. Running it here lets us inspect the engineered columns *before* model training and gives us the curated frames (`train_prep`, `val_prep`, `test_prep`) that feed Feature Store ingestion, baselining, and the training cells below.


In [ ]:
def engineer(df):
    """Apply the production feature pipeline and return (X, y, frame_with_features)."""
    fe = add_derived_columns(winsorize_utilization(df))
    return fe[FEATURES.all], fe[LABEL_COL].to_numpy(), fe

x_train, y_train, train_prep = engineer(train_df)
x_val,   y_val,   val_prep   = engineer(val_df)
x_test,  y_test,  test_prep  = engineer(test_df)

print('Numeric features    :', FEATURES.numeric)
print('Categorical features:', FEATURES.categorical)
print('\nEngineered columns on train_prep:')
train_prep[FEATURES.all].head()


## 5c. Feature Store registration

Register the curated training features in **Amazon SageMaker Feature Store** so the offline store (S3 + Glue table) backs training-set assembly and the online store (low-latency lookup keyed by `beneficiary_id`) backs real-time inference. Both stores share one schema, which is what stops train/serve skew.


In [ ]:
from readmit.features.feature_store import (
    create_or_update_feature_group,
    get_latest_features,
    ingest_features,
    prepare_records,
)

fs_cols = ['beneficiary_id', 'discharge_date', LABEL_COL] + FEATURES.all
fs_frame = train_prep[fs_cols].copy()
fs_records = prepare_records(fs_frame)
fs_records.head()


In [ ]:
if SAGEMAKER_ROLE_ARN:
    fg = create_or_update_feature_group(
        fs_records,
        role_arn=SAGEMAKER_ROLE_ARN,
        name=FEATURE_GROUP_NAME,
        s3_uri_prefix=f's3://{BUCKET}/{PROJECT_PREFIX}/feature-store',
        region=AWS_REGION,
        enable_online_store=True,
        description='30-day readmission features (AAI-540 Group 2 / ClearPath Health Analytics).',
    )
    ingest_features(fs_records, name=FEATURE_GROUP_NAME, region=AWS_REGION)

    # Tabular registration summary
    registration = pd.DataFrame(
        [
            ('feature_group',      FEATURE_GROUP_NAME),
            ('region',             AWS_REGION),
            ('record_identifier',  'beneficiary_id'),
            ('event_time',         'discharge_date'),
            ('records_ingested',   f'{len(fs_records):,}'),
            ('schema_columns',     len(fs_records.columns)),
            ('online_store',       'enabled'),
            ('offline_s3_prefix',  f's3://{BUCKET}/{PROJECT_PREFIX}/feature-store'),
        ],
        columns=['property', 'value'],
    )
    print('Feature Store registration')
    display(registration)

    # Online-store read-back as a single-row DataFrame
    sample_id = str(fs_records['beneficiary_id'].iloc[0])
    online_row = get_latest_features([sample_id], name=FEATURE_GROUP_NAME, region=AWS_REGION)
    print(f'\nOnline-store read-back for beneficiary_id = {sample_id}')
    display(online_row)
else:
    print('Skipping Feature Store registration — SAGEMAKER_ROLE_ARN is not set.')


## 5d. Offline-store training-set assembly via Athena

Now that the FeatureGroup is registered, the offline store is queryable as a Glue-cataloged Athena table — same data the SageMaker Pipeline (Section 9) uses to assemble training sets. Ad-hoc analytical SQL over the curated features, joined to the label, with the guarantee that the same row a model trained on can be retrieved from the online store at inference time.

We do three things here:

1. Resolve the auto-generated Athena table name from the FeatureGroup.
2. Pull the engineered features back as a DataFrame (just like a Pipeline ProcessingStep would).
3. Run one analytical query — readmission rate by `age_band` × `primary_dx_chapter` — to show offline-store usability.


In [ ]:
if SAGEMAKER_ROLE_ARN:
    from readmit.features.feature_store import query_offline_features
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.session import Session as _SmSession
    import boto3 as _boto3

    # Resolve the Athena table auto-created by Feature Store.
    _sm_sess  = _SmSession(boto_session=_boto3.Session(region_name=AWS_REGION))
    _fg       = FeatureGroup(name=FEATURE_GROUP_NAME, sagemaker_session=_sm_sess)
    _athena   = _fg.athena_query()
    OFFLINE_DB    = _athena.database
    OFFLINE_TABLE = _athena.table_name
    ATHENA_OUTPUT = f's3://{BUCKET}/{PROJECT_PREFIX}/athena-results/'
    print(f'Offline-store Athena table: "{OFFLINE_DB}"."{OFFLINE_TABLE}"')
    print(f'Athena results staged at  : {ATHENA_OUTPUT}')

    # Pull a training-set slice — same shape a Pipeline ProcessingStep would build.
    train_assembly_sql = f'''
        SELECT beneficiary_id, {", ".join(FEATURES.all)}, {LABEL_COL}
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        LIMIT 5000
    '''
    offline_train = query_offline_features(
        query=train_assembly_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )
    print(f'\nAthena returned {len(offline_train):,} rows × {offline_train.shape[1]} columns')
    offline_train.head(5)
else:
    print('Skipping Athena offline-store query — SAGEMAKER_ROLE_ARN is not set.')


In [ ]:
if SAGEMAKER_ROLE_ARN:
    # Readmission rate by age band x diagnosis chapter
    cohort_sql = f'''
        SELECT
            age_band,
            primary_dx_chapter,
            COUNT(*)                                AS n_encounters,
            SUM({LABEL_COL})                        AS n_readmits,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)), 4) AS readmit_rate
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        GROUP BY age_band, primary_dx_chapter
        ORDER BY age_band, readmit_rate DESC
    '''
    cohort = query_offline_features(
        query=cohort_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )
    print(f'Cohort breakdown — {len(cohort)} (age_band x primary_dx_chapter) cells')
    cohort
else:
    print('Skipping Athena cohort query — SAGEMAKER_ROLE_ARN is not set.')


### 5d.1 Analytical queries against the offline store

Five queries that exercise the offline store the way different stakeholders would —
each one is a question we could answer **without retraining or touching the online endpoint**.
The point is to demonstrate that the feature store is a *governed, queryable system of record*,
not just a training-data dump.

| # | Audience              | Question                                                                                |
|---|-----------------------|-----------------------------------------------------------------------------------------|
| 1 | Care-ops              | How does readmission risk concentrate across a composite-utilization decile, and what is the lift over the base rate? |
| 2 | Clinical leadership   | Which age × sex × prior-utilization cells carry the highest **lift** vs. the population? |
| 3 | Data engineering      | Numeric coverage (nulls, zeros, p50/p95/p99 → winsorization check) *and* categorical coverage (cardinality, top value, null %). |
| 4 | ML / MLOps            | Readmit-rate trend per encounter month — a drift-surveillance baseline for Model Monitor. |
| 5 | Targeting analyst     | Top-100 worklist ranked by a **cohort-derived risk score** (model-free, computed in SQL). |


#### Q1 — Risk concentration by utilization decile, with lift vs. overall base rate. Composite utilization score weights recent inpatient stays most heavily.

In [ ]:
if SAGEMAKER_ROLE_ARN:    
    q1_sql = f'''
        WITH scored AS (
            SELECT
                {LABEL_COL},
                length_of_stay,
                prior_inpatient_90d,
                prior_ed_90d,
                charlson_index,
                NTILE(10) OVER (
                    ORDER BY (
                        CAST(prior_inpatient_90d AS DOUBLE)
                        + CAST(prior_ed_90d AS DOUBLE) * 0.5
                        + CAST(charlson_index AS DOUBLE)
                    )
                ) AS util_decile
            FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
            WHERE is_deleted = false
        ),
        base AS (
            SELECT AVG(CAST({LABEL_COL} AS DOUBLE)) AS base_rate FROM scored
        )
        SELECT
            util_decile,
            COUNT(*)                                                   AS n_encounters,
            SUM({LABEL_COL})                                           AS n_readmits,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)), 4)                 AS readmit_rate,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)) / b.base_rate, 2)   AS lift_vs_base,
            ROUND(AVG(CAST(prior_inpatient_90d AS DOUBLE)), 2)         AS avg_prior_inpatient_90d,
            ROUND(AVG(CAST(length_of_stay AS DOUBLE)), 2)              AS avg_length_of_stay,
            ROUND(AVG(CAST(charlson_index AS DOUBLE)), 2)              AS avg_charlson_index
        FROM scored
        CROSS JOIN base b
        GROUP BY util_decile, b.base_rate
        ORDER BY util_decile
    '''

    q1 = query_offline_features(
        query=q1_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )

    print(f'Q1 — Risk concentration across utilization deciles ({len(q1)} rows)')
    if len(q1) and 'lift_vs_base' in q1.columns:
        print(f'  Top-decile lift vs. base rate: {q1["lift_vs_base"].iloc[-1]:.2f}x')
    else:
        print('  WARNING: Q1 returned 0 rows. The offline store table is empty.')
        print('  Likely causes:')
        print('    1. Feature group was just (re)created — offline-store ingestion has not landed in S3+Glue yet (allow ~10–15 min after PutRecord).')
        print('    2. A prior cleanup dropped the Glue table; re-run §5b ingestion to repopulate.')
        print(f'    3. Glue table not pointing at the expected location: db="{OFFLINE_DB}", table="{OFFLINE_TABLE}".')
    display(q1)
else:
    print('Skipping Q1 — SAGEMAKER_ROLE_ARN is not set.')

  #### Q2 — Highest-lift age x sex x prior-utilization cells (lift = cell rate / overall rate). Requires n_encounters >= 50 so we ignore tiny cohorts whose rates are noise.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    q2_sql = f'''
        WITH base AS (
            SELECT AVG(CAST({LABEL_COL} AS DOUBLE)) AS base_rate
            FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
            WHERE is_deleted = false
        ),
        risk_cells AS (
            SELECT
                age_band,
                sex,
                CASE
                    WHEN prior_inpatient_90d = 0 THEN 'no_prior'
                    WHEN prior_inpatient_90d = 1 THEN '1_prior'
                    WHEN prior_inpatient_90d BETWEEN 2 AND 3 THEN '2-3_prior'
                    ELSE '4+_prior'
                END                                                AS prior_inpatient_tier,
                COUNT(*)                                           AS n_encounters,
                SUM({LABEL_COL})                                   AS n_readmits,
                AVG(CAST({LABEL_COL} AS DOUBLE))                   AS readmit_rate
            FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
            WHERE is_deleted = false
            GROUP BY 1, 2, 3
        )
        SELECT
            age_band,
            sex,
            prior_inpatient_tier,
            n_encounters,
            n_readmits,
            ROUND(readmit_rate, 4)                                  AS readmit_rate,
            ROUND(readmit_rate / b.base_rate, 2)                    AS lift_vs_base
        FROM risk_cells
        CROSS JOIN base b
        WHERE n_encounters >= 50
        ORDER BY lift_vs_base DESC
        LIMIT 25
    '''

    q2 = query_offline_features(
        query=q2_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )

    print('Q2 — Top-25 highest-lift age x sex x prior-utilization cells (n >= 50)')
    display(q2)
else:
    print('Skipping Q2 — SAGEMAKER_ROLE_ARN is not set.')


##### Q3a — Numeric feature coverage + distribution (p50/p95/p99 expose winsorization tails).

In [ ]:
if SAGEMAKER_ROLE_ARN:    
    numeric_feats = [c for c in FEATURES.numeric if c != LABEL_COL]
    numeric_parts = []
    for col in numeric_feats:
        numeric_parts.append(f"""
        SELECT
            '{col}'                                                          AS feature,
            COUNT(*)                                                         AS n_rows,
            SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END)                   AS n_nulls,
            SUM(CASE WHEN {col} = 0 THEN 1 ELSE 0 END)                       AS n_zeros,
            ROUND(MIN(CAST({col} AS DOUBLE)), 4)                             AS min_v,
            ROUND(APPROX_PERCENTILE(CAST({col} AS DOUBLE), 0.50), 4)         AS p50_v,
            ROUND(AVG(CAST({col} AS DOUBLE)), 4)                             AS mean_v,
            ROUND(APPROX_PERCENTILE(CAST({col} AS DOUBLE), 0.95), 4)         AS p95_v,
            ROUND(APPROX_PERCENTILE(CAST({col} AS DOUBLE), 0.99), 4)         AS p99_v,
            ROUND(MAX(CAST({col} AS DOUBLE)), 4)                             AS max_v
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        """)
    q3a_sql = "\nUNION ALL\n".join(numeric_parts) + "\nORDER BY feature"

    q3a = query_offline_features(
        query=q3a_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )
    q3a['null_pct'] = (q3a['n_nulls'] / q3a['n_rows']).round(4)
    q3a['zero_pct'] = (q3a['n_zeros'] / q3a['n_rows']).round(4)
    print(f'Q3a — Numeric coverage + distribution across {len(q3a)} features')
    display(q3a[['feature', 'n_rows', 'null_pct', 'zero_pct',
                 'min_v', 'p50_v', 'mean_v', 'p95_v', 'p99_v', 'max_v']])

#### Q3b — Categorical coverage (cardinality, top value, top share, null %). Two passes per column: top value (in pandas, off a small grouped pull) plus a single overall coverage query. Keeps Athena SQL simple.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    coverage_parts = []
    for col in FEATURES.categorical:
        coverage_parts.append(f"""
        SELECT
            '{col}'                                                          AS feature,
            COUNT(*)                                                         AS n_rows,
            COUNT(DISTINCT {col})                                            AS n_distinct,
            SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END)                   AS n_nulls
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        """)
    q3b_coverage_sql = "\nUNION ALL\n".join(coverage_parts) + "\nORDER BY feature"
    q3b = query_offline_features(
        query=q3b_coverage_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )

    # Pull the top value + its count per categorical column with one small query.
    top_rows = []
    for col in FEATURES.categorical:
        top_sql = f'''
            SELECT '{col}' AS feature,
                    CAST({col} AS VARCHAR) AS top_value,
                    COUNT(*) AS top_value_count
            FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
            WHERE is_deleted = false AND {col} IS NOT NULL
            GROUP BY {col}
            ORDER BY COUNT(*) DESC
            LIMIT 1
        '''
        top_rows.append(query_offline_features(
            query=top_sql, name=FEATURE_GROUP_NAME,
            region=AWS_REGION, output_location=ATHENA_OUTPUT,
        ))
    q3b = q3b.merge(pd.concat(top_rows, ignore_index=True), on='feature', how='left')
    q3b['null_pct']        = (q3b['n_nulls'] / q3b['n_rows']).round(4)
    q3b['top_value_share'] = (q3b['top_value_count'] / q3b['n_rows']).round(4)

    print(f'\nQ3b — Categorical coverage across {len(q3b)} features')
    display(q3b[['feature', 'n_rows', 'n_distinct', 'null_pct',
                 'top_value', 'top_value_share']])
else:
    print('Skipping Q3b — SAGEMAKER_ROLE_ARN is not set.')


#### Q4 — Readmit-rate trend per encounter month (drift surveillance baseline). event_time is stored as epoch seconds and reflects the source encounter date, so this query is the analytical predecessor to Model Monitor.

In [ ]:
if SAGEMAKER_ROLE_ARN:    
    q4_sql = f'''
        SELECT
            DATE_FORMAT(
                FROM_UNIXTIME(CAST(event_time AS BIGINT)),
                '%Y-%m'
            )                                                       AS encounter_month,
            COUNT(*)                                                AS n_records,
            COUNT(DISTINCT beneficiary_id)                          AS n_distinct_patients,
            SUM({LABEL_COL})                                        AS n_readmits,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)), 4)              AS readmit_rate,
            ROUND(AVG(CAST(length_of_stay AS DOUBLE)), 2)           AS avg_length_of_stay,
            ROUND(AVG(CAST(prior_inpatient_90d AS DOUBLE)), 2)      AS avg_prior_inpatient_90d
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        GROUP BY 1
        ORDER BY encounter_month
    '''

    q4 = query_offline_features(
        query=q4_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )

    print(f'Q4 — Readmit-rate trend over {len(q4)} encounter months (drift baseline)')
    if len(q4):
        overall_rate = q4['n_readmits'].sum() / q4['n_records'].sum()
        q4 = q4.assign(rate_delta_vs_overall=(q4['readmit_rate'] - overall_rate).round(4))
        print(f'  Overall readmit rate    : {overall_rate:.4f}')
        print(f'  Max monthly delta vs avg: {q4["rate_delta_vs_overall"].abs().max():.4f}')
    display(q4)

    if len(q4):
        fig, ax1 = plt.subplots(figsize=(10, 3))
        ax1.bar(q4['encounter_month'], q4['n_records'], alpha=0.4, label='records')
        ax1.set_ylabel('records', color='steelblue')
        ax1.tick_params(axis='x', rotation=45)
        ax2 = ax1.twinx()
        ax2.plot(q4['encounter_month'], q4['readmit_rate'],
                 color='crimson', marker='o', label='readmit rate')
        ax2.axhline(overall_rate, color='crimson', linestyle='--', alpha=0.5,
                    label=f'overall {overall_rate:.3f}')
        ax2.set_ylabel('readmit rate', color='crimson')
        ax1.set_title('Encounter volume + monthly readmit rate (drift surveillance baseline)')
        ax2.legend(loc='upper right')
        plt.tight_layout()
else:
    print('Skipping Q4 — SAGEMAKER_ROLE_ARN is not set.')


  #### Q5 — Worklist ranked by a cohort-derived risk score. The "risk" is the observed readmit rate of the patient's(age_band, sex, prior_inpatient_tier) cohort — model-free, but a defensible heuristic to score encounters before training is even done.

In [ ]:
if SAGEMAKER_ROLE_ARN:  
    q5_sql = f'''
        WITH tiered AS (
            SELECT
                beneficiary_id,
                event_time,
                age_band,
                sex,
                primary_dx_chapter,
                prior_inpatient_90d,
                prior_ed_90d,
                length_of_stay,
                charlson_index,
                {LABEL_COL},
                CASE
                    WHEN prior_inpatient_90d = 0 THEN 'no_prior'
                    WHEN prior_inpatient_90d = 1 THEN '1_prior'
                    WHEN prior_inpatient_90d BETWEEN 2 AND 3 THEN '2-3_prior'
                    ELSE '4+_prior'
                END AS prior_inpatient_tier
            FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
            WHERE is_deleted = false
        ),
        cohort_rates AS (
            SELECT
                age_band,
                sex,
                prior_inpatient_tier,
                AVG(CAST({LABEL_COL} AS DOUBLE)) AS cohort_risk,
                COUNT(*)                         AS cohort_n
            FROM tiered
            GROUP BY 1, 2, 3
        )
        SELECT
            t.beneficiary_id,
            FROM_UNIXTIME(CAST(t.event_time AS BIGINT))   AS event_ts,
            t.age_band,
            t.sex,
            t.primary_dx_chapter,
            t.prior_inpatient_tier,
            t.prior_inpatient_90d,
            t.prior_ed_90d,
            t.length_of_stay,
            t.charlson_index,
            ROUND(c.cohort_risk, 4)                       AS cohort_risk_score,
            c.cohort_n,
            t.{LABEL_COL}                                 AS observed_readmit
        FROM tiered t
        JOIN cohort_rates c
          ON t.age_band = c.age_band
         AND t.sex = c.sex
         AND t.prior_inpatient_tier = c.prior_inpatient_tier
        WHERE c.cohort_n >= 50
        ORDER BY
            c.cohort_risk DESC,
            t.prior_inpatient_90d DESC,
            t.event_time DESC
        LIMIT 100
    '''

    q5 = query_offline_features(
        query=q5_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )

    print('Q5 — Top-100 worklist ranked by cohort-derived risk score')
    if len(q5):
        precision_at_100 = q5['observed_readmit'].mean()
        print(f'  Observed readmit rate within this worklist: {precision_at_100:.4f}')
        print(f'  Highest cohort risk score on the list     : {q5["cohort_risk_score"].max():.4f}')
    display(q5.head(20))
    if len(q5) > 20:
        print(f'... {len(q5) - 20} more rows in q5')
else:
    print('Skipping Q5 — SAGEMAKER_ROLE_ARN is not set.')


## 6a. Baseline — logistic regression

In [ ]:
class _Args:
    learning_rate=0.1; max_depth=4; n_estimators=400
    subsample=0.9; colsample_bytree=0.9; reg_lambda=1.0; seed=42

lr_pipeline = build_logreg_pipeline(_Args()).fit(x_train, y_train)
lr_proba    = lr_pipeline.predict_proba(x_val)[:, 1]
lr_metrics  = compute_metrics(y_val, lr_proba)

print('Logistic-regression validation metrics:')
lr_metrics_df = (
    pd.DataFrame(lr_metrics.items(), columns=['metric', 'value'])
      .assign(value=lambda d: d['value'].astype(float).round(4))
)
lr_metrics_df


## 6b. Primary model — XGBoost

In [ ]:
pos = int(y_train.sum()); neg = int(len(y_train) - pos)
scale_pos_weight = max(neg / max(pos, 1), 1.0)

xgb_pipeline = build_xgb_pipeline(_Args(), scale_pos_weight=scale_pos_weight).fit(x_train, y_train)
xgb_proba    = xgb_pipeline.predict_proba(x_val)[:, 1]
xgb_metrics  = compute_metrics(y_val, xgb_proba)
xgb_metrics['recall_at_top_10pct'] = recall_at_k(y_val, xgb_proba, 0.10)
xgb_metrics['recall_at_top_20pct'] = recall_at_k(y_val, xgb_proba, 0.20)

# Class-balance summary + validation metrics as side-by-side tables.
balance_df = pd.DataFrame(
    [
        ('positives (pos)',  pos),
        ('negatives (neg)',  neg),
        ('scale_pos_weight', round(scale_pos_weight, 3)),
    ],
    columns=['property', 'value'],
)
print('Class imbalance')
display(balance_df)

print('\nXGBoost validation metrics:')
xgb_metrics_df = (
    pd.DataFrame(xgb_metrics.items(), columns=['metric', 'value'])
      .assign(value=lambda d: d['value'].astype(float).round(4))
)
xgb_metrics_df

## 7. Subgroup / fairness evaluation

In [ ]:
subgroup = subgroup_metrics(
    val_prep, y_val, xgb_proba,
    group_cols=['age_band','sex','primary_dx_chapter'],
)
subgroup[['group','value','n','n_pos','auc','pr_auc','precision','recall']]

## 8. Final test-set evaluation + deploy gate

The AUC gate (`AUC_THRESHOLD_DEPLOY = 0.75`) is the same threshold enforced by the SageMaker Pipeline `ConditionStep` and the CI/CD workflow.

In [ ]:
test_proba = xgb_pipeline.predict_proba(x_test)[:, 1]
test_metrics = compute_metrics(y_test, test_proba)
test_metrics['recall_at_top_10pct'] = recall_at_k(y_test, test_proba, 0.10)
test_metrics['recall_at_top_20pct'] = recall_at_k(y_test, test_proba, 0.20)

passed = test_metrics['auc'] >= AUC_THRESHOLD_DEPLOY

# Test-set metrics table.
print('XGBoost test-set metrics:')
test_metrics_df = (
    pd.DataFrame(test_metrics.items(), columns=['metric', 'value'])
      .assign(value=lambda d: d['value'].astype(float).round(4))
)
display(test_metrics_df)

# Deploy-gate verdict table.
gate_df = pd.DataFrame(
    [
        ('test_auc',              round(float(test_metrics['auc']), 4)),
        ('auc_threshold_deploy',  AUC_THRESHOLD_DEPLOY),
        ('margin',                round(float(test_metrics['auc']) - AUC_THRESHOLD_DEPLOY, 4)),
        ('verdict',               'PASS' if passed else 'FAIL'),
        ('note',                  '' if passed else 'synthetic data may not meet target; real CMS data is expected to.'),
    ],
    columns=['property', 'value'],
)
print(f'\nAUC gate (>= {AUC_THRESHOLD_DEPLOY})')
gate_df

## 8b. Model Store — package + register the trained model

Persist the locally-trained XGBoost pipeline as a SageMaker **ModelPackage** in the `ReadmitRiskModels` ModelPackageGroup. Versions enter as `PendingManualApproval`; the AUC-gate helper auto-approves only when the held-out test AUC clears `AUC_THRESHOLD_DEPLOY` (= 0.75). CD then deploys the most recent `Approved` version — never an un-gated one.


In [ ]:
if SAGEMAKER_ROLE_ARN:
    import joblib, tarfile, tempfile, pathlib
    from readmit.models.registry import (
        auto_approve_if_above_threshold,
        ensure_model_package_group,
        list_versions,
        register_model_version,
    )

    ensure_model_package_group(
        group_name=MODEL_PACKAGE_GROUP,
        region=AWS_REGION,
    )

    # Bundle the trained pipeline into model.tar.gz with feature spec + metrics.
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = pathlib.Path(tmp)
        joblib.dump(xgb_pipeline, tmp_path / 'model.joblib')
        (tmp_path / 'feature_spec.json').write_text(json.dumps({
            'numeric': FEATURES.numeric,
            'categorical': FEATURES.categorical,
            'label': LABEL_COL,
        }, indent=2))
        (tmp_path / 'evaluation.json').write_text(json.dumps(test_metrics, indent=2))

        archive = tmp_path / 'model.tar.gz'
        with tarfile.open(archive, 'w:gz') as tar:
            for name in ('model.joblib', 'feature_spec.json', 'evaluation.json'):
                tar.add(tmp_path / name, arcname=name)

        model_s3 = sess.upload_data(
            str(archive), bucket=BUCKET,
            key_prefix=f'{PROJECT_PREFIX}/model-store/artifacts',
        )
        metrics_s3 = sess.upload_data(
            str(tmp_path / 'evaluation.json'), bucket=BUCKET,
            key_prefix=f'{PROJECT_PREFIX}/model-store/metrics',
        )

    # Register the version (always starts as PendingManualApproval).
    pkg_arn = register_model_version(
        model_data_s3_uri=model_s3,
        group_name=MODEL_PACKAGE_GROUP,
        region=AWS_REGION,
        metrics_s3_uri=metrics_s3,
        description=f"Notebook-trained XGBoost — test AUC {test_metrics['auc']:.4f}",
        customer_metadata={
            'trained_in': 'notebook',
            'train_rows': len(x_train),
            'split_ratio': '40/30/30',
            'auc_threshold_deploy': AUC_THRESHOLD_DEPLOY,
        },
    )
    print('Registered ModelPackageArn:', pkg_arn)

    # Auto-approve only when test AUC clears the gate.
    approved = auto_approve_if_above_threshold(
        pkg_arn, test_metrics['auc'],
        threshold=AUC_THRESHOLD_DEPLOY, region=AWS_REGION,
    )
    print('Auto-approval result      :', 'APPROVED' if approved else 'REJECTED')

    versions = list_versions(group_name=MODEL_PACKAGE_GROUP, region=AWS_REGION)
    pd.DataFrame([{
        'arn'    : v['ModelPackageArn'].split('/')[-1],
        'status' : v['ModelApprovalStatus'],
        'created': v['CreationTime'],
    } for v in versions[:10]])
else:
    print('Skipping Model Store registration — SAGEMAKER_ROLE_ARN is not set.')


## 9. (Optional) Run the production SageMaker Pipeline

Skip this section if you only want to train locally. Requires `SAGEMAKER_ROLE_ARN` to be set.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    import boto3
    from botocore.exceptions import WaiterError
    from readmit.pipeline.sagemaker_pipeline import build_pipeline

    pipeline = build_pipeline(
        role_arn=SAGEMAKER_ROLE_ARN,
        pipeline_name='ReadmitRiskPipeline',
        model_package_group='ReadmitRiskModels',
        data_source=DATA_SOURCE,
        s3_data_uri=S3_CURATED_URI,
        n_patients=N_PATIENTS,
        region=AWS_REGION,
        auc_threshold=AUC_THRESHOLD_DEPLOY,
    )
    pipeline.upsert(role_arn=SAGEMAKER_ROLE_ARN)
    execution = pipeline.start()
    print('Pipeline started:', execution.arn)

    try:
        execution.wait()
        print('Pipeline succeeded.')
    except WaiterError:       
        sm = boto3.client('sagemaker', region_name=AWS_REGION)
        steps = sm.list_pipeline_execution_steps(
            PipelineExecutionArn=execution.arn
        ).get('PipelineExecutionSteps', [])
        print(f'\nPipeline FAILED. Step summary ({len(steps)} step(s)):')
        for s in steps:
            print(f"  - {s['StepName']:<30} {s['StepStatus']}")
        for s in [s for s in steps if s['StepStatus'] == 'Failed']:
            print(f"\n--- {s['StepName']} ---")
            print('FailureReason:', s.get('FailureReason') or '(none on step)')
            md = s.get('Metadata') or {}
            if 'ProcessingJob' in md:
                job = md['ProcessingJob'].get('Arn', '').rsplit('/', 1)[-1]
                if job:
                    desc = sm.describe_processing_job(ProcessingJobName=job)
                    print('ProcessingJob:', job)
                    print('  FailureReason:', desc.get('FailureReason') or '(none)')
                    print('  ExitMessage  :', desc.get('ExitMessage') or '(none)')
            elif 'TrainingJob' in md:
                job = md['TrainingJob'].get('Arn', '').rsplit('/', 1)[-1]
                if job:
                    desc = sm.describe_training_job(TrainingJobName=job)
                    print('TrainingJob:', job)
                    print('  FailureReason:', desc.get('FailureReason') or '(none)')
            elif 'RegisterModel' in md:
                print('RegisterModel ARN:', md['RegisterModel'].get('Arn'))
        print('\n(WaiterError suppressed — see step detail above.)')
else:
    print('Skipping SageMaker Pipeline run — SAGEMAKER_ROLE_ARN is not set.')

## 10. Deploy a real-time endpoint (notebook-driven path)

For the classroom demo we deploy the locally-trained XGBoost pipeline as a SageMaker model + real-time endpoint with **data capture enabled**, so Model Monitor has traffic to evaluate.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    import time, joblib, tarfile, tempfile
    from sagemaker.session import Session
    from sagemaker.sklearn.model import SKLearnModel
    from sagemaker.model_monitor import DataCaptureConfig
    from sagemaker.utils import name_from_base

    MODEL_NAME_PREFIX = 'ReadmitRisk-sklearn'
    model_name        = name_from_base(MODEL_NAME_PREFIX)

    if sess is None:
        sess = Session(boto_session=boto3.Session(region_name=AWS_REGION))

    capture_uri = f's3://{BUCKET}/{PROJECT_PREFIX}/data-capture/{ENDPOINT_NAME}/'

    # Reuse the artifact built in 8b if it's still in scope; otherwise build a fresh one.
    if 'model_s3' in globals() and model_s3:
        s3_model_uri = model_s3
        print('Reusing model artifact from Model Store cell:', s3_model_uri)
    else:
        with tempfile.TemporaryDirectory() as tmp:
            joblib.dump(xgb_pipeline, os.path.join(tmp, 'model.joblib'))
            tar_path = os.path.join(tmp, 'model.tar.gz')
            with tarfile.open(tar_path, 'w:gz') as t:
                t.add(os.path.join(tmp, 'model.joblib'), arcname='model.joblib')
            s3_model_uri = sess.upload_data(
                tar_path, bucket=BUCKET,
                key_prefix=f'{PROJECT_PREFIX}/model-store/artifacts',
            )
        print('Uploaded fresh model artifact:', s3_model_uri)

    sk_model = SKLearnModel(
        name=model_name,
        model_data=s3_model_uri,
        role=SAGEMAKER_ROLE_ARN,
        entry_point='readmit/models/inference.py',
        source_dir='./src',
        framework_version='1.2-1', py_version='py3',
        sagemaker_session=sess,
    )
    print(f'Model object name        : {model_name}')

    # Make redeploys idempotent: SageMaker's CreateEndpointConfig rejects an
    # existing name (no upsert). Tear down any prior endpoint + endpoint-config
    # named ENDPOINT_NAME so deploy() can recreate both cleanly.
    sm_pre = boto3.client('sagemaker', region_name=AWS_REGION)
    try:
        sm_pre.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print(f'Deleting existing endpoint     : {ENDPOINT_NAME}')
        sm_pre.delete_endpoint(EndpointName=ENDPOINT_NAME)
        sm_pre.get_waiter('endpoint_deleted').wait(
            EndpointName=ENDPOINT_NAME,
            WaiterConfig={'Delay': 15, 'MaxAttempts': 40},
        )
    except sm_pre.exceptions.ClientError as exc:
        if 'Could not find' not in str(exc):
            raise
    try:
        sm_pre.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
        print(f'Deleted stale endpoint config  : {ENDPOINT_NAME}')
    except sm_pre.exceptions.ClientError as exc:
        if 'Could not find' not in str(exc):
            raise

    # Deploy model and poll for the Create / InService status.
    sk_model.deploy(
        initial_instance_count=1,
        instance_type='ml.m5.large',
        endpoint_name=ENDPOINT_NAME,
        data_capture_config=DataCaptureConfig(
            enable_capture=True, sampling_percentage=100,
            destination_s3_uri=capture_uri,
            sagemaker_session=sess,
        ),
        wait=False,
    )
    print(f'Endpoint create requested : {ENDPOINT_NAME}')
    print('Typical first-deploy time : 5-10 min (container pull + provision + health probes).')
    print('Polling EndpointStatus every 30 s (max 15 min) — safe to interrupt.\n')

    sm_client  = boto3.client('sagemaker', region_name=AWS_REGION)
    max_wait_s = 15 * 60
    started    = time.time()
    status     = 'Creating'
    while True:
        desc    = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        status  = desc['EndpointStatus']
        elapsed = int(time.time() - started)
        print(f'  [{elapsed:>4}s] EndpointStatus = {status}')
        if status in ('InService', 'Failed', 'OutOfService'):
            break
        if elapsed >= max_wait_s:
            print(f'\n  Stopped polling after {max_wait_s // 60} min — endpoint still {status}.')
            print('  Deploy is NOT cancelled. Re-check with:')
            print(f"    boto3.client('sagemaker', region_name=AWS_REGION).describe_endpoint(EndpointName='{ENDPOINT_NAME}')['EndpointStatus']")
            break
        time.sleep(30)

    if status == 'InService':
        print(f'\nEndpoint deployed: {ENDPOINT_NAME}')
    elif status == 'Failed':
        print('\nDeploy FAILED. Reason:', desc.get('FailureReason', '(none)'))
else:
    print('Skipping endpoint deploy — SAGEMAKER_ROLE_ARN is not set.')


## 11. Model Monitor — baseline + hourly drift schedule

Builds a baseline statistics + constraints set from the training data and schedules an hourly drift check against captured inference traffic.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    from datetime import datetime, timedelta, timezone
    from readmit.monitoring.setup_model_monitor import (
        create_data_quality_baseline, create_data_quality_schedule,
    )

    # Pinned so cleanup (§14) and the §11b resume cell can find the schedule
    # without re-running Section 11.
    MONITOR_SCHEDULE_NAME = 'readmit-data-quality-hourly'

    baseline_local = '/tmp/readmit_monitor_baseline.csv'
    train_prep[FEATURES.all].to_csv(baseline_local, index=False)
    baseline_s3 = sess.upload_data(
        baseline_local, bucket=BUCKET,
        key_prefix=f'{PROJECT_PREFIX}/monitor/baseline-input',
    )
    baseline_out   = f's3://{BUCKET}/{PROJECT_PREFIX}/monitor/baseline-output'
    reports_prefix = f's3://{BUCKET}/{PROJECT_PREFIX}/monitor/reports'

    monitor = create_data_quality_baseline(
        role_arn=SAGEMAKER_ROLE_ARN,
        baseline_dataset_s3_uri=baseline_s3,
        output_s3_uri=baseline_out,
        region=AWS_REGION,
    )
    create_data_quality_schedule(
        monitor, endpoint_name=ENDPOINT_NAME,
        schedule_name=MONITOR_SCHEDULE_NAME,
        output_s3_uri=reports_prefix,
    )

    # Hourly cron is cron(0 * ? * * *) — fires at minute :00 of every hour, UTC.
    # AWS runs this server-side; the kernel does NOT need to stay alive.
    now_utc      = datetime.now(timezone.utc).replace(microsecond=0)
    next_run_utc = (now_utc + timedelta(hours=1)).replace(minute=0, second=0)

    schedule_df = pd.DataFrame(
        [
            ('schedule_name',   MONITOR_SCHEDULE_NAME),
            ('endpoint',        ENDPOINT_NAME),
            ('cron',            'cron(0 * ? * * *)  — hourly at :00 UTC'),
            ('baseline_out',    baseline_out),
            ('reports_prefix',  reports_prefix),
            ('now_utc',         now_utc.isoformat()),
            ('next_run_utc',    next_run_utc.isoformat()),
            ('mins_until_run',  int((next_run_utc - now_utc).total_seconds() // 60)),
            ('wait_required',   'No. Shut the kernel down; come back after next_run_utc and run §11b.'),
        ],
        columns=['property', 'value'],
    )
    print('Model Monitor schedule created')
    display(schedule_df)
else:
    print('Skipping Model Monitor setup — SAGEMAKER_ROLE_ARN is not set.')


### 11b. Resume — check the latest scheduled monitoring run

The hourly cron runs **server-side** on AWS at minute `:00 UTC`. You do **not** need to keep this kernel alive — shut Studio down, come back any time after `next_run_utc` (printed in §11), and run only this cell to inspect the latest `MonitoringExecution` + the CloudWatch alarm state for the seven alarms wired up in §12.

This cell is intentionally self-contained: it falls back to the project defaults if `MONITOR_SCHEDULE_NAME` / `ENDPOINT_NAME` / `AWS_REGION` aren't in scope, so a fresh kernel that only ran §1 (Configuration) can still answer the question.


In [ ]:
# Self-contained — safe to run after a fresh kernel.
# Only needs: MONITOR_SCHEDULE_NAME, ENDPOINT_NAME, AWS_REGION (defaults below).
import boto3
import pandas as pd
from IPython.display import display

MONITOR_SCHEDULE_NAME = globals().get('MONITOR_SCHEDULE_NAME', 'readmit-data-quality-hourly')
ENDPOINT_NAME         = globals().get('ENDPOINT_NAME',         'readmit-risk-dev')
AWS_REGION            = globals().get('AWS_REGION',            'us-east-1')

_sm = boto3.client('sagemaker',  region_name=AWS_REGION)
_cw = boto3.client('cloudwatch', region_name=AWS_REGION)

# 1. Schedule state
desc = _sm.describe_monitoring_schedule(MonitoringScheduleName=MONITOR_SCHEDULE_NAME)
last = desc.get('LastMonitoringExecutionSummary') or {}
schedule_state = pd.DataFrame(
    [
        ('schedule_name',    MONITOR_SCHEDULE_NAME),
        ('status',           desc['MonitoringScheduleStatus']),
        ('last_modified',    desc['LastModifiedTime'].isoformat()),
        ('last_exec_status', last.get('MonitoringExecutionStatus', '(no executions yet)')),
        ('last_exec_time',   str(last.get('ScheduledTime', '(no executions yet)'))),
        ('failure_reason',   last.get('FailureReason', '') or ''),
    ],
    columns=['property', 'value'],
)
print('Monitoring schedule state')
display(schedule_state)

# 2. Execution history (most recent 10)
execs = _sm.list_monitoring_executions(
    MonitoringScheduleName=MONITOR_SCHEDULE_NAME, MaxResults=10,
).get('MonitoringExecutionSummaries', [])
if execs:
    exec_df = pd.DataFrame([{
        'scheduled_time': e['ScheduledTime'].isoformat(),
        'status':         e['MonitoringExecutionStatus'],
        'creation_time':  e['CreationTime'].isoformat(),
        'failure_reason': e.get('FailureReason', '') or '',
    } for e in execs])
    print(f'\nMost recent {len(exec_df)} executions')
    display(exec_df)
else:
    print('\nNo executions have fired yet — hourly cron runs at minute :00 UTC.')

# 3. CloudWatch alarm state for the seven §12 alarms
alarm_names = [f'Readmit-{ENDPOINT_NAME}-{s}' for s in (
    'Latency-p95', '5xx', '4xx', 'CPU',
    'DataDrift', 'ModelQuality-AUC', 'Custom-AUC-Floor',
)]
alarms = _cw.describe_alarms(AlarmNames=alarm_names).get('MetricAlarms', [])
alarm_df = pd.DataFrame(
    [(a['AlarmName'], a['StateValue'], (a.get('StateReason') or '')[:80]) for a in alarms],
    columns=['alarm', 'state', 'reason'],
) if alarms else pd.DataFrame(columns=['alarm', 'state', 'reason'])
print(f'\nCloudWatch alarm state ({len(alarm_df)} / {len(alarm_names)} found)')
display(alarm_df)


## 12. CloudWatch alarms + SNS alerting


In [ ]:
from readmit.monitoring.alerts import build_dashboard, configure_alarms

topic_arn = configure_alarms(
    endpoint_name=ENDPOINT_NAME,
    region=AWS_REGION,
)
dashboard_name = build_dashboard(
    endpoint_name=ENDPOINT_NAME,
    region=AWS_REGION,
)
print('Alarm topic ARN :', topic_arn)
print('Dashboard       :', dashboard_name)
print(f'Dashboard URL   : https://{AWS_REGION}.console.aws.amazon.com/cloudwatch/home?region={AWS_REGION}#dashboards:name={dashboard_name}')

## 13. Live inference test

In [ ]:
if SAGEMAKER_ROLE_ARN:
    import boto3
    smr = boto3.client('sagemaker-runtime', region_name=AWS_REGION)
    payload = {'instances': [{
        'age': 72, 'sex': 'F', 'primary_diagnosis': 'circulatory',
        'discharge_disposition': 'home', 'payer_type': 'medicare_ffs',
        'length_of_stay': 5, 'n_chronic_conditions': 4, 'charlson_index': 3,
        'prior_inpatient_90d': 1, 'prior_ed_90d': 2, 'prior_outpatient_90d': 3,
    }]}
    resp = smr.invoke_endpoint(
        EndpointName=ENDPOINT_NAME, ContentType='application/json',
        Accept='application/json', Body=json.dumps(payload),
    )
    print(json.loads(resp['Body'].read()))
else:
    # Local invocation through the same code path the endpoint serves.
    from readmit.models.inference import input_fn, predict_fn, output_fn
    payload = '{"instances": [{"age":72,"sex":"F","primary_diagnosis":"circulatory","discharge_disposition":"home","payer_type":"medicare_ffs","length_of_stay":5,"n_chronic_conditions":4,"charlson_index":3,"prior_inpatient_90d":1,"prior_ed_90d":2,"prior_outpatient_90d":3}]}'
    x   = input_fn(payload, 'application/json')
    out, _ = output_fn(predict_fn(x, xgb_pipeline), 'application/json')
    print(out)


## 14. Cleanup — tear down every AWS resource this notebook created

Run only when you are done. The cell below is **idempotent** (safe to re-run)
and gated behind `CLEANUP = False` so it never fires on a kernel restart.

**Two-phase workflow:**

1. **Preview (always runs).** With `CLEANUP = False`, the cell resolves every
   real resource name — including dynamic ones (`Model` objects matching the
   `ReadmitRisk-sklearn-*` prefix, every registered `ModelPackage` version, and
   per-prefix S3 object counts) — and prints them in a `planned_df` table.
   Use this after the §11 hourly monitor run + §11b execution table look healthy.
2. **Delete (gated).** Flip `CLEANUP = True` and re-run to actually delete, in
   order:
   1. **Model Monitor schedules** (`readmit-data-quality-hourly`,
      `readmit-model-quality-daily`) — must be stopped before the endpoint goes away
   2. **CloudWatch alarms** (7: `Latency-p95`, `5xx`, `4xx`, `CPU`, `DataDrift`,
      `ModelQuality-AUC`, `Custom-AUC-Floor`) and the **dashboard**
      (`Readmit-{ENDPOINT_NAME}`)
   3. **SNS topic** `readmit-model-alerts`
   4. **SageMaker endpoint**, **endpoint config**, and matching **`Model` object(s)**
   5. **SageMaker Pipeline** `ReadmitRiskPipeline` (stops any running executions first)
   6. Every **`ModelPackage` version**, then the **`ModelPackageGroup`** `ReadmitRiskModels`
   7. **Feature Group** `readmit-encounter-features` — drops the online store
      and Glue Data Catalog table (offline-store S3 objects are removed in step 8)
   8. **Project S3 prefixes** under `s3://{BUCKET}/{PROJECT_PREFIX}/`:
      `curated/`, `feature-store/`, `athena-results/`, `data-capture/`, `monitor/`,
      and the registered `model.tar.gz` artifacts

Every delete is wrapped so a missing resource does not abort the rest of the
teardown. A summary table of what was deleted (or skipped) is printed at the end.


In [ ]:
CLEANUP = False  # Flip to True to actually delete every AWS resource this notebook created.

import boto3
from botocore.exceptions import ClientError

# ---- Lifted constants — single source of truth for preview AND delete ----
DATA_QUALITY_SCHEDULE  = 'readmit-data-quality-hourly'
MODEL_QUALITY_SCHEDULE = 'readmit-model-quality-daily'
ALARM_SUFFIXES         = ('Latency-p95', '5xx', '4xx', 'CPU',
                          'DataDrift', 'ModelQuality-AUC', 'Custom-AUC-Floor')
ALARM_NAMES            = [f'Readmit-{ENDPOINT_NAME}-{s}' for s in ALARM_SUFFIXES]
DASHBOARD_NAME         = f'Readmit-{ENDPOINT_NAME}'
SNS_TOPIC_BASENAME     = 'readmit-model-alerts'
MODEL_NAME_PREFIX      = 'ReadmitRisk-sklearn'   # SKLearnModel.deploy() prefix in §10
PIPELINE_NAME          = 'ReadmitRiskPipeline'
S3_PREFIXES            = [f'{PROJECT_PREFIX}/{p}/' for p in
                          ('curated', 'feature-store', 'athena-results',
                           'data-capture', 'monitor', 'models')]

# ---- Read-only preview — always runs so you can validate before deleting --
sm  = boto3.client('sagemaker',  region_name=AWS_REGION)
sts = boto3.client('sts',        region_name=AWS_REGION)
s3r = boto3.resource('s3',       region_name=AWS_REGION)

try:
    account_id = sts.get_caller_identity()['Account']
    sns_topic_arn = f'arn:aws:sns:{AWS_REGION}:{account_id}:{SNS_TOPIC_BASENAME}'
except Exception:
    sns_topic_arn = f'arn:aws:sns:{AWS_REGION}:<unknown>:{SNS_TOPIC_BASENAME}'

# Resolve dynamic Model object names (§10 emits 'ReadmitRisk-sklearn-<ts>')
try:
    resolved_models = [m['ModelName']
                       for page in sm.get_paginator('list_models').paginate(NameContains=MODEL_NAME_PREFIX)
                       for m in page['Models']]
except ClientError:
    resolved_models = []

# Resolve dynamic ModelPackage versions
try:
    resolved_packages = [v['ModelPackageArn'].rsplit('/', 1)[-1]
                         for page in sm.get_paginator('list_model_packages').paginate(
                             ModelPackageGroupName=MODEL_PACKAGE_GROUP)
                         for v in page.get('ModelPackageSummaryList', [])]
except ClientError:
    resolved_packages = []

# Resolve S3 object counts per prefix (read-only HEAD-equivalent — no delete here)
s3_counts: dict[str, int] = {}
bucket_obj = s3r.Bucket(BUCKET)
for p in S3_PREFIXES:
    try:
        s3_counts[p] = sum(1 for _ in bucket_obj.objects.filter(Prefix=p).limit(10_000))
    except Exception:
        s3_counts[p] = -1   # error sentinel

preview_rows: list[tuple[str, str]] = []
preview_rows.append(('MonitoringSchedule', DATA_QUALITY_SCHEDULE))
preview_rows.append(('MonitoringSchedule', MODEL_QUALITY_SCHEDULE))
for a in ALARM_NAMES:
    preview_rows.append(('CloudWatchAlarm', a))
preview_rows.append(('CloudWatchDashboard', DASHBOARD_NAME))
preview_rows.append(('SNSTopic', sns_topic_arn))
preview_rows.append(('Endpoint', ENDPOINT_NAME))
preview_rows.append(('EndpointConfig', ENDPOINT_NAME))
if resolved_models:
    for m in resolved_models:
        preview_rows.append(('Model', m))
else:
    preview_rows.append(('Model', f'(none matching "{MODEL_NAME_PREFIX}")'))
preview_rows.append(('Pipeline', PIPELINE_NAME))
if resolved_packages:
    for v in resolved_packages:
        preview_rows.append(('ModelPackage', v))
else:
    preview_rows.append(('ModelPackage', f'(none in group "{MODEL_PACKAGE_GROUP}")'))
preview_rows.append(('ModelPackageGroup', MODEL_PACKAGE_GROUP))
preview_rows.append(('FeatureGroup', FEATURE_GROUP_NAME))
for p in S3_PREFIXES:
    n = s3_counts[p]
    label = f's3://{BUCKET}/{p}  ({"ERROR" if n < 0 else f"{n}+ objects" if n >= 10_000 else f"{n} objects"})'
    preview_rows.append(('S3Prefix', label))

planned_df = pd.DataFrame(preview_rows, columns=['kind', 'name'])
mode = 'DELETE (CLEANUP=True)' if (CLEANUP and SAGEMAKER_ROLE_ARN) else 'PREVIEW only (read-only)'
print(f'Cleanup plan — {mode}')
print(f'Account={account_id if "account_id" in dir() else "?"}  Region={AWS_REGION}  Bucket={BUCKET}')
display(planned_df)

# ---- Destructive section (gated) -----------------------------------------
if CLEANUP and SAGEMAKER_ROLE_ARN:
    cw  = boto3.client('cloudwatch', region_name=AWS_REGION)
    sns = boto3.client('sns',        region_name=AWS_REGION)

    teardown_log: list[tuple[str, str, str]] = []

    def _record(kind: str, name: str, status: str) -> None:
        teardown_log.append((kind, name, status))
        print(f'  [{status:>7}] {kind:<22} {name}')

    def _safe(kind: str, name: str, fn,
              *, ok_codes=('ResourceNotFound', 'ValidationException', 'NoSuchEntity', '404')):
        try:
            fn()
            _record(kind, name, 'deleted')
        except ClientError as exc:
            code = exc.response.get('Error', {}).get('Code', '')
            if any(c in code or c in str(exc) for c in ok_codes):
                _record(kind, name, 'absent')
            else:
                _record(kind, name, f'ERROR:{code}')
        except Exception as exc:
            _record(kind, name, f'ERROR:{type(exc).__name__}')

    print('\n[1/8] Model Monitor schedules')
    for sch in (DATA_QUALITY_SCHEDULE, MODEL_QUALITY_SCHEDULE):
        _safe('MonitoringSchedule', sch,
              lambda s=sch: sm.delete_monitoring_schedule(MonitoringScheduleName=s))

    print('\n[2/8] CloudWatch alarms + dashboard')
    _safe('CloudWatchAlarms', f'{len(ALARM_NAMES)} alarms',
          lambda: cw.delete_alarms(AlarmNames=ALARM_NAMES))
    _safe('CloudWatchDashboard', DASHBOARD_NAME,
          lambda: cw.delete_dashboards(DashboardNames=[DASHBOARD_NAME]))

    print('\n[3/8] SNS topic')
    _safe('SNSTopic', sns_topic_arn,
          lambda: sns.delete_topic(TopicArn=sns_topic_arn))

    print('\n[4/8] Endpoint, endpoint config, Model object(s)')
    _safe('Endpoint',       ENDPOINT_NAME,
          lambda: sm.delete_endpoint(EndpointName=ENDPOINT_NAME))
    _safe('EndpointConfig', ENDPOINT_NAME,
          lambda: sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME))
    for mname in resolved_models:
        _safe('Model', mname,
              lambda m=mname: sm.delete_model(ModelName=m))
    if not resolved_models:
        _record('Model', f'(none matching "{MODEL_NAME_PREFIX}")', 'absent')

    print('\n[5/8] SageMaker Pipeline')
    try:
        for page in sm.get_paginator('list_pipeline_executions').paginate(PipelineName=PIPELINE_NAME):
            for ex in page.get('PipelineExecutionSummaries', []):
                if ex.get('PipelineExecutionStatus') in ('Executing', 'Stopping'):
                    _safe('PipelineExecution', ex['PipelineExecutionArn'],
                          lambda a=ex['PipelineExecutionArn']:
                              sm.stop_pipeline_execution(PipelineExecutionArn=a))
    except ClientError:
        pass
    _safe('Pipeline', PIPELINE_NAME,
          lambda: sm.delete_pipeline(PipelineName=PIPELINE_NAME))

    print('\n[6/8] Model Registry (versions then group)')
    for pkg in resolved_packages:
        _safe('ModelPackage', pkg,
              lambda a=pkg: sm.delete_model_package(ModelPackageName=a))
    _safe('ModelPackageGroup', MODEL_PACKAGE_GROUP,
          lambda: sm.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP))

    print('\n[7/8] Feature Group')
    _safe('FeatureGroup', FEATURE_GROUP_NAME,
          lambda: sm.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME))

    print('\n[8/8] S3 prefixes under', f's3://{BUCKET}/{PROJECT_PREFIX}/')
    for prefix in S3_PREFIXES:
        try:
            resp = bucket_obj.objects.filter(Prefix=prefix).delete()
            n = sum(len(r.get('Deleted', [])) for r in resp) if resp else 0
            _record('S3Prefix', f's3://{BUCKET}/{prefix}', f'{n} obj')
        except Exception as exc:
            _record('S3Prefix', f's3://{BUCKET}/{prefix}', f'ERROR:{type(exc).__name__}')

    print('\nTeardown summary')
    summary_df = pd.DataFrame(teardown_log, columns=['kind', 'name', 'status'])
    display(summary_df)
    print(f'Total actions: {len(summary_df)}  '
          f'(deleted={int((summary_df.status == "deleted").sum())}, '
          f'absent={int((summary_df.status == "absent").sum())}, '
          f'errors={int(summary_df.status.str.startswith("ERROR").sum())})')
elif CLEANUP:
    print('\nCLEANUP requested but SAGEMAKER_ROLE_ARN is not set — nothing deleted.')
else:
    print('\nPreview only. Review planned_df above, then set CLEANUP = True and re-run to delete.')


## 15. Model Card

A short, summary of what this model is, what it's for, and
where it should *not* be used. Mirrors the Google / HuggingFace model-card
template.

### Model details
- **Name:** `ReadmitRiskModels` (primary version: XGBoost; baseline: logistic regression)
- **Owners:** AAI-540 Group 2 — Jose Sandoval, Manikanta Katuri, Michael Domingo
- **Version:** auto-incremented in the SageMaker Model Registry; only versions
  with held-out test AUC ≥ `AUC_THRESHOLD_DEPLOY` (= 0.75) are auto-approved.
- **License / data use:** the training data is **CMS DE-SynPUF** — a
  publicly-released, **synthesised** sample of 2008–2010 Medicare FFS claims
  that CMS generated from real claims and de-identified for unrestricted
  research / educational use. It contains **no PHI**. This model inherits
  that license.

### Intended use
- **Primary use case:** rank inpatient discharges by 30-day all-cause
  readmission risk so a care-management team can prioritise post-discharge
  outreach (med reconciliation calls, transitional-care visits).
- **Intended users:** care-coordination analysts and clinical informatics
  teams inside a Medicare-focused payer or ACO.
- **Out-of-scope uses:** ❌ individual treatment decisions, ❌ admission /
  discharge denials, ❌ insurance pricing, ❌ any use on pediatric, non-Medicare,
  or non-US populations.

### Training data
- **Source:** CMS DE-SynPUF (Data Entrepreneurs' **Syn**thetic Public Use File)
  in OMOP CDM v5.x, hosted on the AWS Open Data Registry at
  `s3://synpuf-omop/` — 100k-beneficiary tier.
- **Provenance:** CMS produced DE-SynPUF by sampling 5 % of 2008–2010
  Medicare FFS claims and applying a structured synthesis / perturbation
  process so the released records preserve statistical properties (age,
  diagnosis, utilisation distributions) without corresponding to any real
  beneficiary. Switching to real CMS claims (the RIF / LDS files) only
  requires changing `DATA_SOURCE` to point at a private bucket with a signed
  Data Use Agreement; the pipeline code is identical.
- **Window:** claims dated 2008–2010 (the full DE-SynPUF release).
- **Cohort:** adult Medicare beneficiaries with at least one inpatient
  encounter; `length_of_stay` clipped to 1–30 days.
- **Label:** `readmitted_30d = 1` iff a subsequent inpatient admission for the
  same beneficiary occurs within 30 days of the index discharge.
- **Split:** 40 / 30 / 30 train / test / validation, **patient-grouped** and
  time-aware (see `src/readmit/data/splits.py`).

### Features
12 model columns declared in `FeatureSpec` (`src/readmit/config.py`) and
served from the **SageMaker Feature Store** group `readmit-encounter-features`:

- **Numeric (6):** `length_of_stay`, `prior_inpatient_90d`, `prior_ed_90d`,
  `prior_outpatient_90d`, `n_chronic_conditions`, `charlson_index`
- **Categorical (6):** `age_band`, `sex`, `primary_dx_chapter`,
  `discharge_disposition`, `payer_type`, `los_bucket`

Feature definitions and the engineering pipeline are versioned in
`src/readmit/features/`. Same row a model trained on is retrievable from the
online store at inference time (offline-store Athena view is shown in §5d).

### Metrics & evaluation
- **Primary:** AUC-ROC (deploy gate ≥ 0.75)
- **Secondary:** PR-AUC, Recall@top-10%, Recall@top-20%, Brier score
- **Fairness:** per-subgroup AUC / PR-AUC / precision / recall across
  `age_band`, `sex`, and `primary_dx_chapter` (Section 7). Cells with
  fewer than 30 rows or a single label are dropped to keep estimates stable.
- Last-run numbers are persisted next to the model artifact in the registry
  as `evaluation.json`.

### Known limitations
1. **DE-SynPUF is a synthesised sample, not live claims.** It is statistically
   representative of 2008–2010 Medicare FFS but is not a real patient
   population. Real-world AUC is expected to be in the same ballpark but is
   unverified until the model is retrained on actual CMS RIF / LDS data.
2. **Payer mix is heuristic.** DE-SynPUF doesn't expose payer detail in the
   1k / 100k tiers, so `payer_type` is derived deterministically from
   `person_id` (≈ 55 % FFS / 30 % MA / 15 % dual). This will not match any
   real plan's distribution and should be replaced with the payer file on
   real claims.
3. **Diagnosis chapters are bucketed.** Primary diagnosis is hashed into 8
   stable chapters (`primary_dx_chapter`) because the OMOP concept dictionary
   isn't shipped with the open dataset. A production deployment should swap
   this for an ICD-10 → AHRQ-CCS mapping.
4. **No social-determinants features.** Income, language, transportation,
   and prior medication adherence are not in DE-SynPUF; they materially
   affect real readmission risk and would change subgroup behaviour.
5. **Distribution shift.** The 2008–2010 training window predates COVID-era
   utilisation patterns; expect drift on contemporary data and rely on
   Model Monitor (Section 11) to surface it.

### Ethical / clinical considerations
- The model surfaces a *risk score*, not a diagnosis. Care teams retain full
  clinical judgement.
- Subgroup metrics are reported in Section 7 so reviewers can check that
  performance does not collapse for any demographic.
- Because DE-SynPUF is de-identified by construction, no consent, BAA, or
  IRB review is required for this prototype. A production version on real
  CMS RIF / LDS claims would require a signed DUA, encrypted-at-rest storage,
  IAM-scoped access, and an IRB-style review for clinical deployment.

### Maintenance / retraining cadence
- **Scheduled retrain:** monthly via the CD workflow (`.github/workflows/cd.yml`)
  running the SageMaker Pipeline `ReadmitRiskPipeline`.
- **Drift-triggered retrain:** Model Monitor publishes feature-baseline
  drift counts; the `Readmit-{ENDPOINT_NAME}-DataDrift` alarm (one of seven
  configured in §12: `Latency-p95`, `5xx`, `4xx`, `CPU`, `DataDrift`,
  `ModelQuality-AUC`, `Custom-AUC-Floor`, all publishing to the
  `readmit-model-alerts` SNS topic and surfaced on the `Readmit-{ENDPOINT_NAME}`
  CloudWatch dashboard) triggers an ad-hoc pipeline run.
- **Rollback:** previous `Approved` versions stay in the Model Registry;
  the CD job can promote any prior version by ARN.
- **Decommissioning:** Section 14 contains an idempotent teardown that
  removes every AWS resource the notebook created (monitor schedules,
  alarms, dashboard, SNS topic, endpoint stack, pipeline, model-package
  versions and group, feature group, and project S3 prefixes) when
  `CLEANUP = True`.
- **Contact:** open an issue in the GitHub repo; tag `@aai540-group2`.
